# Visualize motion path in recording playback (aligned)

<video controls src="./assets/path_seeking_recording_aligned.webm">


Load the nanotube + methane example recording as an MDAnalsis universe:

In [1]:
from nanover.mdanalysis import universe_from_recording
from nanover.trajectory import FrameData

universe = universe_from_recording("../recordings/nanotube-example-recording.nanover.zip")

Find the centroid of the methane molecule for each frame:

In [2]:
import numpy as np

NANOTUBE_ATOMS = list(range(0, 60))
METHANE_ATOMS = list(range(60, 65))

METHANE_CENTROIDS_ABSOLUTE = []
for i, _ in enumerate(universe.trajectory):
    positions = universe.atoms.positions[METHANE_ATOMS] / 10  # angstrom -> nm
    centroid = np.mean(positions, axis=0)
    METHANE_CENTROIDS_ABSOLUTE.append(centroid)

Approximating the nanotube as a rigid object, find the relative transform between the nanotube's initial position and orientation and it's position and orientation at each frame:

In [3]:
from nanover.utilities.transforms import Transform, find_transformation_between_point_patterns

NANOTUBE_ATOMS = list(range(0, 60))

_ = universe.trajectory[0]
NANOTUBE_INITIAL_POSITIONS = universe.atoms.positions[NANOTUBE_ATOMS] / 10  # angstrom -> nm

NANOTUBE_TRANSFORMS: list[Transform] = []
for _ in universe.trajectory:
    nanotube_positions = universe.atoms.positions[NANOTUBE_ATOMS] / 10  # angstrom -> nm
    matrix = find_transformation_between_point_patterns(NANOTUBE_INITIAL_POSITIONS, nanotube_positions)
    transform = Transform.from_parent_to_local_matrix(matrix)
    NANOTUBE_TRANSFORMS.append(transform)

Set up the server with playback of the universe:

In [4]:
from nanover.app import OmniRunner
from nanover.mdanalysis import UniverseSimulation

simulation = UniverseSimulation.from_universe(universe, name="nanotube + methane")
simulation.playback_factor = 30
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE:  trajectory seeker (aligned)")
imd_runner.load(0)

C:\Users\ragzo\anaconda3\envs\nanover-dev\Lib\site-packages\MDAnalysis\coordinates\base.py:730: UserWarning: Reader has no dt information, set to 1.0 ps
  return self.ts.dt


Import the jupyter utilities for drawing + interaction:

In [5]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

Continuously update a line in the scene that shows the path of the methane centroid relative the current frame's nanotube:

In [6]:
from nanover.jupyter import FrameListener

class RelativePathVisuals(FrameListener):
    points = []

    def on_frame_update(self, full_frame: FrameData, frame_update: FrameData):
        frame_i_transform = NANOTUBE_TRANSFORMS[simulation.prev_frame]

        self.points = []
        for n, (frame_n_centroid, frame_n_transform) in enumerate(zip(METHANE_CENTROIDS_ABSOLUTE, NANOTUBE_TRANSFORMS)):
            # maintaining relative position to the nanotube, where would this centroid be if the nanotube was in its frame 0 position?
            frame_0_centroid = frame_n_transform.point_local_to_parent(frame_n_centroid)
            # maintaining relative position to the nanotube, where would this centroid be if the nanotube was in its current position?
            frame_i_centroid = frame_i_transform.point_parent_to_local(frame_0_centroid)
            self.points.append(frame_i_centroid)
        utilities.objects.update_line("path", positions=self.points, size=0.01)

visuals = RelativePathVisuals.from_runner(imd_runner)
visuals.start()

C:\Users\ragzo\anaconda3\envs\nanover-dev\Lib\site-packages\MDAnalysis\coordinates\base.py:730: UserWarning: Reader has no dt information, set to 1.0 ps
  return self.ts.dt


Define a mode that shows the nearest point on the centroid path as the user controllers hover close to it, and seeks the recording when they click:

In [7]:
from nanover.jupyter import Mode
from nanover.utilities.intersection import closest_point_on_polyline


class SeekLineMode(Mode):
    def on_button_pressed(self, *, key: str, cursor: dict, button: str):
        if button == "primary":
            next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])
            result = closest_point_on_polyline(visuals.points, next_pos)
            utilities.notify_all(f"seek to frame #{result.index}")
            simulation.seek_to_frame_index(result.index)

    def on_cursor_updated(self, *, key: str, cursor: dict):
        next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])
        result = closest_point_on_polyline(visuals.points, next_pos)

        if result.distance < 0.05:
            utilities.objects.update_shape(f"cursor.{key}", position=result.point, size=0.02)
        else:
            utilities.objects.remove_shape(f"cursor.{key}")


utilities.use_interaction_modes()
utilities.add_interaction_mode(SeekLineMode, "seek", icon="⏩")